# Phân tích SQL – Dự đoán phí vận chuyển đơn hàng thương mại điện tử Olist
**Nhóm 8 · Lớp AI2107 · Môn ADY201M · FALL26**

---
## 1. Mục đích của bước SQL

Sau bước làm sạch (Phần 3), nhóm dùng SQL để **truy vấn và tóm tắt trực tiếp dữ liệu nhằm khám phá dữ liệu**. Ở bước này nhóm **chưa biết** yếu tố nào liên quan đến phí, và chưa chọn mô hình; SQL chỉ giúp nhìn dữ liệu trước khi đặt giả thuyết cụ thể cho EDA và mô hình.

**Các câu hỏi khám phá của bước SQL:**

1. Dữ liệu có quy mô và phân bố phí vận chuyển (`freight_value`) như thế nào? (Query 4)
2. Phí khác nhau ra sao giữa các **danh mục, người bán và thời gian**, và có đặc điểm nào của đơn đi kèm với sự khác nhau đó? (Query 1 đến 3)
3. Khi một đặc điểm của đơn thay đổi, phí trung bình có thay đổi theo không? Nhóm xem bốn đặc điểm đo được trong bảng: cân nặng, thể tích, khoảng cách giữa người bán và người mua, giá bán. (Query 5 đến 8)

**Giả thuyết cần kiểm tra (chưa phải kết quả).** Theo hiểu biết về vận chuyển, hàng nặng, cồng kềnh hoặc giao xa thường tốn phí hơn; giá bán có thể liên quan gián tiếp vì hàng đắt thường lớn và nặng. Nhóm chọn bốn đặc điểm trên vì đây là các đặc điểm số có sẵn và hợp lý về mặt thực tế, không phải vì đã biết chúng quyết định phí. Bước SQL chỉ cho biết dữ liệu có phù hợp với các giả thuyết đó hay không.

**Những việc SQL không làm.** SQL chỉ **mô tả**: không kết luận nhân quả và không xếp hạng mức độ quan trọng của các yếu tố vì chúng liên quan với nhau (ví dụ cân nặng và thể tích). Các câu hỏi về mức liên hệ mạnh nhất, khả năng dự đoán (sai số, phần biến động giải thích được) và dạng quan hệ (tuyến tính hay không) cần tương quan, biểu đồ và mô hình, được xét ở EDA (Phần 5) và hồi quy (Phần 6).

## 2. Dữ liệu sử dụng

- **Nguồn:** Brazilian E-Commerce Public Dataset by Olist (Kaggle), giấy phép CC BY-NC-SA 4.0.
- **Bảng dùng trong SQL:** `olist_clean` (108.937 dòng, 10 cột) do Phần 3 xuất ra (tệp `olist_clean.csv`): chỉ giữ đơn đã giao, giai đoạn 01/2017–08/2018, bỏ dòng thiếu tọa độ, cân nặng hoặc kích thước, bỏ dòng có cân nặng bằng 0 và phí vận chuyển bằng 0.
- **Đơn vị quan sát:** một sản phẩm trong một đơn hàng, xác định bởi cặp (`order_id`, `order_item_id`).

| Cột | Ý nghĩa |
|---|---|
| `order_id`, `order_item_id` | Khóa của dòng (đơn hàng, số thứ tự sản phẩm trong đơn) |
| `seller_id` | Mã người bán |
| `order_purchase_timestamp` | Thời điểm đặt hàng (Query 3 lấy tháng từ cột này bằng `strftime`) |
| `price` | Giá bán của sản phẩm (BRL) |
| `weight_g` | Cân nặng sản phẩm (gam) |
| `volume_cm3` | Thể tích sản phẩm (dài × cao × rộng) |
| `distance_km` | Khoảng cách đường chim bay giữa người bán và khách (km), tính theo công thức Haversine từ tọa độ mã bưu chính |
| `category` | Danh mục sản phẩm (tiếng Anh; `unknown` nếu thiếu) |
| `freight_value` | Phí vận chuyển (BRL), **biến cần dự đoán** |

## 3. Chuẩn bị môi trường
Nạp file dữ liệu sạch vào SQLite (cơ sở dữ liệu chạy trong bộ nhớ) và khai báo hàm `sql()` để chạy câu lệnh.

In [1]:
import sqlite3, pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# Tệp do Phần 3 xuất ra. Colab: kéo olist_clean.csv vào /content/ rồi giữ nguyên dòng dưới.
# Trong repo: '../data/cleaned/olist_clean.csv'
CSV_PATH = '/content/olist_clean.csv'
df = pd.read_csv(CSV_PATH)

conn = sqlite3.connect(':memory:')
df.to_sql('olist_clean', conn, index=False, if_exists='replace')

def sql(query):
    """Chạy một câu SQL và trả về bảng kết quả"""
    return pd.read_sql_query(query, conn)

print(df.shape)

(108937, 10)


### Kiểm tra dữ liệu đã nạp
Xác nhận bảng nạp vào đúng với kết quả của bước làm sạch.

In [2]:
print('Số dòng, số cột        :', df.shape)
print('Tổng ô thiếu           :', df.isna().sum().sum())
print('Số dòng trùng khóa      :', df.duplicated(['order_id', 'order_item_id']).sum())
print('Giai đoạn đặt hàng      :', df['order_purchase_timestamp'].str[:7].min(), 'đến', df['order_purchase_timestamp'].str[:7].max())
print('Dòng category = unknown :', (df['category'] == 'unknown').sum())
assert df.shape == (108937, 10), 'Nạp nhầm tệp: cần bản 10 cột, 108.937 dòng'

Số dòng, số cột        : (108937, 10)
Tổng ô thiếu           : 0
Số dòng trùng khóa      : 0
Giai đoạn đặt hàng      : 2017-01 đến 2018-08
Dòng category = unknown : 1510


## 4. Các truy vấn

Gồm **8 truy vấn**, chia thành hai nhóm:

- **Query 1 đến 3 (khám phá theo danh mục, người bán, thời gian):** xem phí trung bình theo danh mục, theo người bán đông nhất và theo tháng.
- **Query 4 đến 8:** Query 4 mô tả quy mô dữ liệu và phí vận chuyển; Query 5 đến 8 lần lượt xem phí theo cân nặng, thể tích, khoảng cách và giá. Các yếu tố được chia thành nhóm bằng mốc tròn, dễ đọc, rồi tính số dòng và phí trung bình. Mốc chia không được chọn để tối ưu kết quả.

Cột `so_dong` là số dòng (sản phẩm trong đơn), không phải số đơn.

---
## Query 1 — Phí theo danh mục

**Câu hỏi phân tích**: Danh mục nào có phí trung bình cao nhất, và có phải do hàng nặng hơn không?
**Mục đích**: Xem chênh lệch phí giữa các danh mục và đối chiếu với cân nặng, khoảng cách. Chỉ xét danh mục từ 1.000 dòng, bỏ `unknown`.

In [3]:
print("=== 1. PHÍ THEO DANH MỤC ===")
res1 = sql("""
SELECT category,
       COUNT(*) AS so_dong,
       ROUND(AVG(freight_value), 2) AS phi_TB,
       ROUND(AVG(weight_g), 2) AS can_nang_TB,
       ROUND(AVG(distance_km), 2) AS khoang_cach_TB
FROM olist_clean
WHERE category <> 'unknown'
GROUP BY category
HAVING COUNT(*) >= 1000
ORDER BY phi_TB DESC
LIMIT 5;
""")
print(res1)

=== 1. PHÍ THEO DANH MỤC ===
              category  so_dong  phi_TB  can_nang_TB  khoang_cach_TB
0     office_furniture     1655   40.17     11329.06          558.63
1  luggage_accessories     1073   27.83      5787.57          555.01
2         garden_tools     4196   22.94      2826.94          649.89
3                 baby     2961   22.24      3271.29          599.65
4                 auto     4121   21.86      2595.50          617.00


**Nhận xét**: Năm danh mục đắt nhất có phí 21,86 đến 40,17 BRL; `office_furniture` đứng đầu vì hàng nặng nhất (11.329 g), trong khi khoảng cách các danh mục gần nhau (555 đến 650 km). Chênh phí giữa danh mục chủ yếu đi cùng cân nặng.

---
## Query 2 — Phí theo người bán

**Câu hỏi phân tích**: Những người bán đông nhất có mức phí trung bình khác nhau không, và vì sao?
**Mục đích**: Xem `seller_id` có liên quan đến phí hay không.

In [4]:
print("=== 2. PHÍ THEO NGƯỜI BÁN ===")
res2 = sql("""
SELECT seller_id AS nguoi_ban,
       COUNT(*) AS so_dong,
       ROUND(AVG(freight_value), 2) AS phi_TB,
       ROUND(AVG(weight_g), 2) AS can_nang_TB,
       ROUND(AVG(distance_km), 2) AS khoang_cach_TB
FROM olist_clean
GROUP BY seller_id
ORDER BY so_dong DESC
LIMIT 5;
""")
print(res2)

=== 2. PHÍ THEO NGƯỜI BÁN ===


                          nguoi_ban  so_dong  phi_TB  can_nang_TB  khoang_cach_TB
0  6560211a19b47992c3666cc44a7e94c0     1991   13.70       255.40          596.67
1  4a3ca9315b744ce9f8e9374361493884     1947   17.61      2140.12          562.80
2  1f50f920176fa81dab994f9023523100     1865   18.75      1801.61          692.51
3  cc419e0650a3c5ba77189a1882b7556a     1704   14.45       336.53          609.89
4  da8622b14eb17ae2831f4ac5b9dab84a     1546   16.07      1333.27          401.48


**Nhận xét**: Năm người bán đông nhất có phí 13,70 đến 18,75 BRL, thấp hơn mức chung 20,01 BRL; người bán hàng nhẹ (255 g và 337 g) có phí thấp nhất. Khác biệt chủ yếu do loại hàng và khoảng cách giao.

---
## Query 3 — Phí theo tháng

**Câu hỏi phân tích**: Phí trung bình có thay đổi theo thời gian không?
**Mục đích**: Kiểm tra phí trung bình có ổn định theo thời gian không.

In [5]:
print("=== 3. PHÍ THEO THÁNG ===")
res3 = sql("""
SELECT STRFTIME('%Y-%m', order_purchase_timestamp) AS thang,
       COUNT(*) AS so_dong,
       ROUND(AVG(freight_value), 2) AS phi_TB,
       ROUND(AVG(weight_g), 2) AS can_nang_TB,
       ROUND(AVG(distance_km), 2) AS khoang_cach_TB
FROM olist_clean
GROUP BY thang
ORDER BY thang;
""")
print(res3)

=== 3. PHÍ THEO THÁNG ===


      thang  so_dong  phi_TB  can_nang_TB  khoang_cach_TB
0   2017-01      912   17.17      2054.11          619.41
1   2017-02     1858   19.92      2469.12          650.44
2   2017-03     2883   19.00      2231.39          644.31
3   2017-04     2548   19.53      2260.64          673.09
4   2017-05     3983   19.34      2233.41          629.33
5   2017-06     3459   19.53      2291.80          578.24
6   2017-07     4393   19.15      2383.11          608.31
7   2017-08     4756   19.03      2180.40          629.60
8   2017-09     4711   19.77      2165.06          648.03
9   2017-10     5178   19.74      2087.72          632.28
10  2017-11     8435   19.52      2114.21          605.70
11  2017-12     6152   18.92      2045.49          586.76
12  2018-01     7998   19.07      2166.15          591.66
13  2018-02     7478   18.58      1909.74          615.02
14  2018-03     7981   20.86      2150.34          593.18
15  2018-04     7634   20.74      2246.12          560.66
16  2018-05   

**Nhận xét**: Từ 02/2017 đến 02/2018, phí trung bình dao động trong khoảng hẹp 18,6 đến 19,9 BRL. Từ 03/2018 phí cao hơn, 19,9 đến 22,95 BRL, cao nhất vào tháng 07/2018. Tháng 01/2017 chỉ có 912 dòng nên không dùng để so sánh.

---
## Query 4 — Tổng quan dữ liệu và phí vận chuyển

**Câu hỏi phân tích**: Dữ liệu có bao nhiêu dòng, bao nhiêu đơn hàng? Phí vận chuyển nhỏ nhất, lớn nhất và trung bình là bao nhiêu?
**Mục đích**: Nắm quy mô dữ liệu và đặc điểm của biến mục tiêu `freight_value` trước khi đi vào từng yếu tố ảnh hưởng.

In [6]:
print("=== 4. TỔNG QUAN DỮ LIỆU VÀ PHÍ VẬN CHUYỂN ===")
print(sql("""
SELECT COUNT(*) AS so_dong,
       COUNT(DISTINCT order_id) AS so_don_duy_nhat,
       MIN(freight_value) AS phi_nho_nhat,
       ROUND(AVG(freight_value), 2) AS phi_ship_TB,
       MAX(freight_value) AS phi_lon_nhat
FROM olist_clean;
"""))

=== 4. TỔNG QUAN DỮ LIỆU VÀ PHÍ VẬN CHUYỂN ===


   so_dong  so_don_duy_nhat  phi_nho_nhat  phi_ship_TB  phi_lon_nhat
0   108937            95377          0.01        20.01        409.68


**Nhận xét**: Dữ liệu có **108.937** dòng thuộc **95.377** đơn hàng. Phí trung bình là 20,01 BRL nhưng phí cao nhất lên tới 409,68 BRL (gấp khoảng **20 lần** mức trung bình), tức phí lệch phải mạnh. Vì vậy khi đánh giá mô hình cần báo MAE cùng sai số theo từng nhóm, không chỉ dựa vào mức trung bình.

---
## Query 5 — Phí ship theo nhóm cân nặng

**Câu hỏi phân tích**: Hàng càng nặng thì phí vận chuyển có tăng không?
**Mục đích**: Mô tả phí theo nhóm cân nặng (`weight_g`).

In [7]:
print("=== 5. PHÍ SHIP THEO NHÓM CÂN NẶNG ===")
print(sql("""
WITH b AS (
    SELECT freight_value,
           CASE WHEN weight_g < 500 THEN '1. Rất nhẹ (<500g)'
               WHEN weight_g < 2000 THEN '2. Nhẹ (500-2000g)'
               WHEN weight_g < 5000 THEN '3. Trung bình (2-5kg)'
               ELSE '4. Nặng (>5kg)' END AS nhom_can_nang
    FROM olist_clean
)
SELECT nhom_can_nang,
       COUNT(*) AS so_dong,
       ROUND(AVG(freight_value), 2) AS phi_ship_TB
FROM b
GROUP BY nhom_can_nang
ORDER BY nhom_can_nang;
"""))

=== 5. PHÍ SHIP THEO NHÓM CÂN NẶNG ===


           nhom_can_nang  so_dong  phi_ship_TB
0     1. Rất nhẹ (<500g)    43532        15.17
1     2. Nhẹ (500-2000g)    40632        17.89
2  3. Trung bình (2-5kg)    11220        22.81
3         4. Nặng (>5kg)    13553        39.62


**Nhận xét**: Phí trung bình tăng từ 15,17 BRL (dưới 500g) lên 39,62 BRL (trên 5kg), gấp **2,6 lần**, và tăng dần qua từng nhóm.

---
## Query 6 — Phí ship theo nhóm thể tích

**Câu hỏi phân tích**: Hàng càng cồng kềnh thì phí vận chuyển có tăng không?
**Mục đích**: Mô tả phí theo nhóm thể tích (`volume_cm3`).

In [8]:
print("=== 6. PHÍ SHIP THEO NHÓM THỂ TÍCH ===")
print(sql("""
WITH b AS (
    SELECT freight_value,
           CASE WHEN volume_cm3 < 2000 THEN '1. Nhỏ (<2.000 cm3)'
               WHEN volume_cm3 < 5000 THEN '2. Vừa (2.000-5.000)'
               WHEN volume_cm3 < 15000 THEN '3. Lớn (5.000-15.000)'
               ELSE '4. Rất lớn (>15.000)' END AS nhom_the_tich
    FROM olist_clean
)
SELECT nhom_the_tich,
       COUNT(*) AS so_dong,
       ROUND(AVG(freight_value), 2) AS phi_ship_TB
FROM b
GROUP BY nhom_the_tich
ORDER BY nhom_the_tich;
"""))

=== 6. PHÍ SHIP THEO NHÓM THỂ TÍCH ===
           nhom_the_tich  so_dong  phi_ship_TB
0    1. Nhỏ (<2.000 cm3)    16927        14.91
1   2. Vừa (2.000-5.000)    30224        16.27
2  3. Lớn (5.000-15.000)    29070        18.61
3   4. Rất lớn (>15.000)    32716        27.36


**Nhận xét**: Phí trung bình tăng từ 14,91 BRL (dưới 2.000 cm3) lên 27,36 BRL (trên 15.000 cm3), gấp **1,8 lần**; mức tăng rõ nhất ở nhóm rất lớn.

---
## Query 7 — Phí ship theo nhóm khoảng cách

**Câu hỏi phân tích**: Khoảng cách giao hàng (người bán đến người mua) ảnh hưởng thế nào đến phí vận chuyển?
**Mục đích**: Mô tả phí theo nhóm khoảng cách (`distance_km`).

In [9]:
print("=== 7. PHÍ SHIP THEO NHÓM KHOẢNG CÁCH ===")
print(sql("""
WITH b AS (
    SELECT freight_value,
           CASE WHEN distance_km < 200 THEN '1. Gần (<200km)'
               WHEN distance_km < 800 THEN '2. Trung bình (200-800km)'
               WHEN distance_km < 1500 THEN '3. Xa (800-1500km)'
               ELSE '4. Rất xa (>1500km)' END AS nhom_khoang_cach
    FROM olist_clean
)
SELECT nhom_khoang_cach,
       COUNT(*) AS so_dong,
       ROUND(AVG(freight_value), 2) AS phi_ship_TB
FROM b
GROUP BY nhom_khoang_cach
ORDER BY nhom_khoang_cach;
"""))

=== 7. PHÍ SHIP THEO NHÓM KHOẢNG CÁCH ===


            nhom_khoang_cach  so_dong  phi_ship_TB
0            1. Gần (<200km)    28096        12.66
1  2. Trung bình (200-800km)    54032        19.80
2         3. Xa (800-1500km)    17099        23.74
3        4. Rất xa (>1500km)     9710        35.91


**Nhận xét**: Phí trung bình tăng từ 12,66 BRL (dưới 200km) lên 35,91 BRL (trên 1500km), gấp **2,8 lần**, và tăng đều qua từng nhóm.

---
## Query 8 — Phí ship theo nhóm giá bán

**Câu hỏi phân tích**: Sản phẩm giá cao hơn thì phí vận chuyển có cao hơn không?
**Mục đích**: Mô tả phí theo nhóm giá (`price`).

In [10]:
print("=== 8. PHÍ SHIP THEO NHÓM GIÁ BÁN ===")
print(sql("""
WITH b AS (
    SELECT freight_value,
           CASE WHEN price < 50 THEN '1. Rẻ (<50)'
               WHEN price < 100 THEN '2. 50-100'
               WHEN price < 200 THEN '3. 100-200'
               ELSE '4. Đắt (>200)' END AS nhom_gia
    FROM olist_clean
)
SELECT nhom_gia,
       COUNT(*) AS so_dong,
       ROUND(AVG(freight_value), 2) AS phi_ship_TB
FROM b
GROUP BY nhom_gia
ORDER BY nhom_gia;
"""))

=== 8. PHÍ SHIP THEO NHÓM GIÁ BÁN ===


        nhom_gia  so_dong  phi_ship_TB
0    1. Rẻ (<50)    37965        14.73
1      2. 50-100    31941        18.01
2     3. 100-200    26093        23.13
3  4. Đắt (>200)    12938        34.16


**Nhận xét**: Phí trung bình tăng từ 14,73 BRL (giá dưới 50) lên 34,16 BRL (giá trên 200), gấp **2,3 lần**. Giá cao thường đi với hàng lớn và nặng, nên đây chưa phải tác động riêng của giá.

---
## 5. Kết luận chung

### Bảng kết luận cho từng query

| Query | Phát hiện | Kết luận |
|---|---|---|
| 1. Danh mục | `office_furniture` đắt nhất (40,17 BRL) và nặng nhất (11.329 g); khoảng cách các danh mục gần nhau | Chênh phí giữa các danh mục chủ yếu đi cùng cân nặng |
| 2. Người bán | 5 người bán đông nhất có phí 13,70 đến 18,75 BRL, thấp hơn mức chung; người bán hàng nhẹ có phí thấp | Khác biệt giữa các người bán chủ yếu do loại hàng và khoảng cách giao |
| 3. Theo tháng | Phí 18,6–19,9 BRL đến 02/2018, rồi 19,9–22,95 BRL, cao nhất tháng 07/2018 | Phí trung bình thay đổi theo tháng, cao hơn từ 03/2018 |
| 4. Tổng quan | Phí trung bình 20,01 BRL, cao nhất 409,68 BRL (gấp khoảng 20 lần): lệch phải | Phí lệch phải mạnh: mức trung bình thấp hơn nhiều so với giá trị lớn nhất |
| 5. Cân nặng | 15,17 → 39,62 BRL (×2,6) | Hàng càng nặng, phí trung bình càng cao |
| 6. Thể tích | 14,91 → 27,36 BRL (×1,8) | Hàng càng lớn, phí trung bình càng cao, rõ nhất ở nhóm rất lớn |
| 7. Khoảng cách | 12,66 → 35,91 BRL (×2,8) | Giao càng xa, phí trung bình càng cao |
| 8. Giá | 14,73 → 34,16 BRL (×2,3); giá đi cùng hàng lớn, nặng | Giá cao đi cùng phí cao, nhưng chưa phải tác động riêng của giá |

### Kết luận chung

1. Cả bốn yếu tố đều đi cùng phí cao hơn. Theo mức tăng giữa nhóm thấp nhất và cao nhất, khoảng cách (×2,8) và cân nặng (×2,6) lớn nhất, rồi giá (×2,3) và thể tích (×1,8). SQL chưa cho biết yếu tố nào quan trọng nhất; việc này thuộc EDA (tương quan) và độ quan trọng của đặc trưng ở Phần 6.
2. Giá cao thường đi với hàng lớn và nặng, nên các bảng theo nhóm chưa phải tác động riêng của từng yếu tố.
3. Phí lệch phải mạnh (trung bình 20,01 BRL, cao nhất 409,68 BRL), nên cần báo MAE kèm sai số theo nhóm.
4. Chênh phí giữa danh mục và người bán chủ yếu do cân nặng và khoảng cách; phí trung bình cũng thay đổi theo tháng, cao hơn từ 03/2018.

Bốn yếu tố này (`weight_g`, `volume_cm3`, `distance_km`, `price`) là các đặc trưng Phần 6 dùng. SQL chỉ cho thấy chúng đi cùng phí; việc chọn đặc trưng và đo mức đóng góp thuộc EDA và mô hình.

## 6. Hạn chế và bước tiếp theo

**Hạn chế:**
- Chỉ so sánh trung bình theo nhóm, không tách được tác động riêng của từng yếu tố khi các yếu tố liên quan với nhau; kết quả chỉ mô tả, chưa phải kết luận về nguyên nhân hay mức độ quan trọng.
- Phí trung bình bị kéo lên bởi một số phí rất lớn (cao nhất 409,68 BRL), nên cần xem thêm trung vị ở EDA.
- Chưa xác định được phí của đơn nhiều sản phẩm là phí riêng của sản phẩm hay phí của cả đơn được chia (Phần 3, mục 9.1).
- Khoảng cách tính từ tọa độ mã bưu chính theo đường chim bay, không phải quãng đường thực.
- Sự thay đổi phí theo tháng (Query 3) chưa có dữ liệu để giải thích nguyên nhân.

**Bước tiếp theo:** EDA bằng Python (Phần 5: biểu đồ, tương quan Pearson giữa các yếu tố) rồi hồi quy (Phần 6: Linear Regression và Random Forest, đo bằng MAE, RMSE, R²).

In [11]:
conn.close()
print('\nĐã đóng kết nối SQLite.')


Đã đóng kết nối SQLite.
